# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Parsing the citation RDD

The citation file is initially read as strings. I have removed the CSV header, split each row, convert the patent IDs to integers, and represent each citation as a tuple of (CITING, CITED).

In [6]:
citation_header = rddCitations.first()

citationsRDD = (rddCitations.filter(lambda line: line != citation_header)
    .map(lambda line: line.split(","))
    .map(lambda fields: (int(fields[0]),int(fields[1]))))

In [7]:
citationsRDD.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

## Parsing the patent RDD

For every patent I preserve the original CSV fields and extract its state. The patent number becomes the RDD key so it can later be used for joins. Empty state values are represented as None.

In [8]:
def parse_patent(line):
    fields = line.split(",")
    patent_id = int(fields[0])
    state = fields[5].replace('"', '').strip()
    if state == "":
        state = None
    return (patent_id,(fields, state))

In [9]:
patent_header = rddPatents.first()

In [11]:
patentsRDD = (
    rddPatents
    .filter(lambda line: line != patent_header)
    .map(parse_patent))

In [12]:
patentsRDD.take(3)

[(3070801,
  (['3070801',
    '1963',
    '1096',
    '',
    '"BE"',
    '""',
    '',
    '1',
    '',
    '269',
    '6',
    '69',
    '',
    '1',
    '',
    '0',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   None)),
 (3070802,
  (['3070802',
    '1963',
    '1096',
    '',
    '"US"',
    '"TX"',
    '',
    '1',
    '',
    '2',
    '6',
    '63',
    '',
    '0',
    '',
    '',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   'TX')),
 (3070803,
  (['3070803',
    '1963',
    '1096',
    '',
    '"US"',
    '"IL"',
    '',
    '1',
    '',
    '2',
    '6',
    '63',
    '',
    '9',
    '',
    '0.3704',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   'IL'))]

## Created the patent state lookup RDD

For the joins I only need the patent number and state, so I extract these values into a key-value RDD and cache it because it will be reused.

In [13]:
patentStates = (patentsRDD.mapValues(lambda value: value[1]).cache())

In [14]:
patentStates.take(5)

[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

The first join needs to look up the state of the cited patent.

I rearranged each citation from (CITING, CITED) into (CITED, CITING).

In [15]:
citationsByCited = citationsRDD.map(lambda citation: (citation[1],citation[0]))

In [16]:
citationsByCited.take(5)

[(956203, 3858241),
 (1324234, 3858241),
 (3398406, 3858241),
 (3557384, 3858241),
 (3634889, 3858241)]

## Join citations with the cited patent state

left-joind citations with the patent-state RDD using the cited patent as the key.

In [17]:
citationsWithCitedState = (citationsByCited.leftOuterJoin(patentStates))

reorganized the records so that the citing patent becomes the key.

In [18]:
citationsByCiting = citationsWithCitedState.map(
    lambda record: (record[1][0],record[1][1]))

In [19]:
citationStatePairs = (citationsByCiting.join(patentStates).cache())

In [20]:
citationStatePairs.count()

16516270

I retaind records only when both states exist and the cited state's value is equal to the citing patent's state.

In [21]:
sameStateRDD = citationStatePairs.filter(
    lambda record:
        record[1][0] is not None
        and record[1][1] is not None
        and record[1][0] == record[1][1])

In [22]:
sameStateRDD.take(10)

[(4259430, ('NY', 'NY')),
 (4259430, ('NY', 'NY')),
 (4259430, ('NY', 'NY')),
 (4259430, ('NY', 'NY')),
 (4259430, ('NY', 'NY')),
 (4064343, ('IN', 'IN')),
 (4064343, ('IN', 'IN')),
 (3914880, ('MA', 'MA')),
 (4831647, ('IL', 'IL')),
 (4831647, ('IL', 'IL'))]

## Counting same-state citations for each patent

I mapped each record to a count of one and use reduceByKey to sum the counts for each citing patent.

In [23]:
sameStateCountsRDD = (sameStateRDD.map(lambda record: (record[0],1)).reduceByKey(operator.add))

In [24]:
sameStateCountsRDD.take(10)

[(4259430, 5),
 (4064343, 2),
 (3914880, 1),
 (4831647, 4),
 (4465134, 13),
 (3930822, 3),
 (4758234, 3),
 (4152408, 2),
 (4391721, 1),
 (4101744, 5)]

## Adding the same-state count to the original patent records

I left-joind the counts with the original patent dataset so patents without same-state citations are retained. Missing counts are converted to zero.

In [26]:
patentRows = patentsRDD.mapValues(lambda value: value[0])

In [27]:
augmentedPatentsRDD = (
    patentRows
    .leftOuterJoin(sameStateCountsRDD)
    .map(
        lambda record:
            record[1][0] +
            [
                0 if record[1][1] is None
                else record[1][1]]))

## top ten patents

ten records with the largest same-state citation counts.

In [28]:
top_rdd = augmentedPatentsRDD.takeOrdered(10,key=lambda row: -row[-1])

In [30]:
for row in top_rdd:
    print(row)

['5959466', '1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5998655', '1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5936426', '1999', '14466', '1997', '"US"', '"CA